# 06 · matplotlib 製圖（基礎）

用 matplotlib 把資料與幾何形狀畫成靜態圖，掌握繪圖介面、常用圖型、形狀繪製、圖面標註與中文字型設定。

## 學習目標
- 分辨狀態式 state-based（plt.xxx）與物件式 object-oriented（fig, ax）兩種繪圖介面，並能在物件式介面上作圖。
- 控制圖的尺寸與解析度（figsize / dpi），並正確使用 show / savefig / close / tight_layout 管理輸出與資源。
- 選用合適的常用圖型：散點圖 scatter、折線圖 plot、長條圖 bar、直方圖 histogram。
- 用 patches 與 collections 繪製幾何多邊形，並以等比例座標 set_aspect equal 維持形狀正確。
- 為圖面加上完整標註：標題、軸標籤、圖例 legend、文字 text、格線 grid 與軸的開關。
- 設定中文字型 font，讓圖中中文與負號正常顯示。

In [ ]:
# 中文字型設定（本書會畫圖才需要）
import os
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
for fp in ['NotoSansCJK-Regular.otf', 'msjh.ttc', 'mingliu.ttc']:
    try:
        if os.path.exists(fp):
            fm.fontManager.addfont(fp)
    except Exception:
        pass
plt.rcParams['font.family'] = ['Microsoft JhengHei', 'Noto Sans CJK TC', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False

## 兩種繪圖介面與圖物件

matplotlib 有兩種作圖方式。狀態式介面 state-based interface 用 `plt.plot`、`plt.title` 等函式，對「目前這張圖」連續下指令，適合快速試畫。物件式介面 object-oriented interface 則先取得 figure 與 axes 兩個物件，再對它們呼叫方法。

心智模型：figure 是整塊畫布，axes 是畫布上的一個座標系（一張子圖）。`fig, ax = plt.subplots()` 一次拿到兩者。

為什麼學物件式：一張畫布上有多個座標系時，狀態式難以指明對象；物件式可控、可重用，正式圖較不易出錯。本書後續一律以物件式為主。

形狀：
```
fig, ax = plt.subplots()
ax.plot(x, y)
```

In [ ]:
# 概念：同一條折線分別用狀態式與物件式畫出，對照兩種寫法
import numpy as np

# 造一小串數列當示範資料
x = np.arange(0, 6)                 # 0 到 5 共 6 個整數
y = np.array([1, 3, 2, 5, 4, 6])    # 對應的 y 值（自造）

# 狀態式：對「目前的圖」連續下指令
plt.plot(x, y)                      # 在當前 figure 上畫線
plt.title('狀態式介面')             # 設定當前圖的標題
plt.show()                          # 顯示並結束這張圖

# 物件式：先取得 fig 與 ax，再對 ax 下指令
fig, ax = plt.subplots()            # fig 是畫布，ax 是座標系
ax.plot(x, y)                       # 在指定的 ax 上畫線
ax.set_title('物件式介面')          # 對該 ax 設定標題（注意：方法名是 set_title）
plt.show()

print('fig 型別：', type(fig).__name__)
print('ax  型別：', type(ax).__name__)

## 圖的尺寸、輸出與資源管理

圖的實際畫素由兩個參數共同決定：圖尺寸 figsize（單位英吋，寬高）與解析度 dpi（每英吋畫素數）。輸出畫素約為 figsize × dpi，dpi 越高越清晰、檔案越大。

輸出與資源相關方法：

| 方法 | 作用 |
|---|---|
| show | 在畫面上顯示圖 |
| savefig | 把圖存成檔案（可指定 dpi） |
| close | 關閉圖、釋放記憶體 |
| tight_layout | 自動調整邊距，避免標籤被裁切 |

為什麼學：批次作圖時若不 `close`，圖物件會累積佔用記憶體；`tight_layout` 則避免軸標籤或標題被切掉。

In [ ]:
# 概念：同一張圖以不同 figsize 與 dpi 各存一份，比較尺寸後關閉釋放
x = np.linspace(0, 10, 100)         # 0 到 10 取 100 個等距點
y = np.sin(x)                       # 對每點取 sin（向量化，無需迴圈）

settings = [(4, 3, 80), (8, 6, 150)]   # 各組 (寬英吋, 高英吋, dpi)
for w, h, d in settings:
    fig, ax = plt.subplots(figsize=(w, h), dpi=d)   # figsize 與 dpi 共同決定畫素
    ax.plot(x, y)
    ax.set_title('正弦曲線')
    fig.tight_layout()              # 自動排版，避免標籤被裁切
    fname = f'sine_{w}x{h}_{d}dpi.png'
    fig.savefig(fname)              # 存檔，沿用該圖的 dpi
    # 注意：批次作圖務必 close，否則圖物件累積佔記憶體
    plt.close(fig)                  # 關閉並釋放這張圖
    px_w, px_h = w * d, h * d       # 估算輸出畫素
    print(f'{fname}: 約 {px_w} x {px_h} 畫素')

print('已存檔的圖檔：', [f for f in os.listdir('.') if f.startswith('sine_')])

## 常用圖型

選圖原則：資料長什麼樣，就配什麼圖。

| 圖型 | 方法 | 適用 | 常用參數 |
|---|---|---|---|
| 折線圖 plot | ax.plot | 看趨勢、有序變化 | color、linestyle |
| 散點圖 scatter | ax.scatter | 看兩變數分布與關係 | color、marker、s |
| 長條圖 bar | ax.bar | 比較類別數值 | color、width |
| 直方圖 histogram | ax.hist | 看單一數值的分布 | bins、color |

判斷力：折線強調趨勢、散點看分布、長條比類別、直方看數值分布。

In [ ]:
# 概念：在 2x2 子座標各畫一種圖型並列對照
rng = np.random.default_rng(0)      # 固定亂數種子，結果可重現

# 自造示範資料
t = np.arange(12)                   # 12 個時間點
trend = np.cumsum(rng.normal(0, 1, 12))   # 累積和模擬一條趨勢
px = rng.normal(0, 1, 50)           # 散點的 x（50 點）
py = px * 0.8 + rng.normal(0, 0.5, 50)    # 與 px 相關的 y
cats = ['A', 'B', 'C', 'D']         # 四個類別名稱
vals = [5, 9, 4, 7]                 # 各類別數值
samples = rng.normal(50, 10, 300)   # 300 筆數值，看其分布

fig, axes = plt.subplots(2, 2, figsize=(9, 7))   # 2x2 共四個座標系
axes[0, 0].plot(t, trend)                        # 折線：看趨勢
axes[0, 0].set_title('折線圖 plot')
axes[0, 1].scatter(px, py, s=15)                 # 散點：看分布（s 是點大小）
axes[0, 1].set_title('散點圖 scatter')
axes[1, 0].bar(cats, vals)                       # 長條：比類別
axes[1, 0].set_title('長條圖 bar')
axes[1, 1].hist(samples, bins=20)                # 直方：看數值分布（bins 是分組數）
axes[1, 1].set_title('直方圖 histogram')
fig.tight_layout()
plt.show()
plt.close(fig)

## 幾何多邊形繪製

matplotlib 不只畫資料，也能畫幾何圖形。常用元件：

| 元件 | 用途 |
|---|---|
| patches.Rectangle | 矩形（左下角座標 + 寬高） |
| patches.Polygon | 多邊形（一串頂點座標） |
| collections.LineCollection | 一次畫多條線段的集合 |

把形狀加入座標系要用 `ax.add_patch`（單一形狀）或 `ax.add_collection`（線段集合）。

關鍵：`ax.set_aspect('equal')` 讓 x、y 軸等比例，正方形、圓形與角度才不會被拉扁變形。

In [ ]:
# 概念：自造一個多邊形與幾條線段，畫出後設等比例座標確認不變形
from matplotlib import patches
from matplotlib.collections import LineCollection

# 自造一個五邊形的頂點座標
poly_pts = np.array([[1, 1], [3, 1], [4, 3], [2, 4], [0, 3]])

# 自造幾條線段，每條是 [(起點), (終點)]
segments = [[(0, 0), (5, 0)], [(5, 0), (5, 5)], [(0, 0), (0, 5)]]

fig, ax = plt.subplots(figsize=(5, 5))
rect = patches.Rectangle((0.5, 0.5), 1, 1, facecolor='lightgray')   # 左下角(0.5,0.5)、寬高各1
ax.add_patch(rect)                                   # 把矩形加入座標系
polygon = patches.Polygon(poly_pts, closed=True, facecolor='skyblue', edgecolor='navy')
ax.add_patch(polygon)                                # 把多邊形加入座標系
lines = LineCollection(segments, colors='red', linewidths=2)
ax.add_collection(lines)                             # 把線段集合加入座標系
ax.set_xlim(-1, 6)                                   # add_patch 不會自動縮放，需手動設範圍
ax.set_ylim(-1, 6)
# 技巧：set_aspect('equal') 是維持形狀正確的關鍵
ax.set_aspect('equal')                               # x、y 同比例，形狀不被拉扁
ax.set_title('幾何多邊形與線段')
plt.show()
plt.close(fig)
print('多邊形頂點數：', len(poly_pts))

## 圖面標註與座標控制

標註讓圖能被獨立讀懂，不必再口頭解釋。常用標註：

| 標註 | 方法 | 說明 |
|---|---|---|
| 標題 title | ax.set_title | 圖的主題 |
| 軸標籤 xlabel/ylabel | ax.set_xlabel/set_ylabel | 說明各軸代表什麼 |
| 圖例 legend | ax.legend | 依各線的 label 自動生成 |
| 文字 text | ax.text | 在指定資料座標放文字 |
| 格線 grid | ax.grid | 輔助對齊讀值 |

重點：`legend` 依賴畫圖時給的 `label`；`text` 以資料座標定位；純幾何圖常用 `ax.axis('off')` 關閉座標軸。

In [ ]:
# 概念：取一張折線圖補上完整標註
x = np.linspace(0, 10, 50)
y1 = np.sin(x)
y2 = np.cos(x)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, y1, label='sin')         # label 供 legend 使用
ax.plot(x, y2, label='cos')
ax.set_title('正弦與餘弦')          # 標題
ax.set_xlabel('x 值')               # x 軸標籤
ax.set_ylabel('函數值')             # y 軸標籤
ax.legend()                         # 依 label 生成圖例
ax.grid(True)                       # 開格線輔助讀值
# 技巧：text 用資料座標定位，這裡標在 (x=5, y=1) 附近
ax.text(5, 1.0, '交會區', color='purple')
plt.show()
plt.close(fig)

## 中文字型設定

matplotlib 預設字型不含中文字符，直接畫中文會變成方框（俗稱豆腐字）；而預設的負號 minus 也可能在某些字型下顯示為亂碼。

解法兩步：
1. 用字型管理 font_manager 的 `addfont` 註冊一個含中文的字型檔，再透過全域設定 `rcParams['font.family']` 指定字型族。
2. 設 `rcParams['axes.unicode_minus'] = False`，改用一般 ASCII 減號，負號才不會亂碼。

本書第二個 cell 已做過全域設定，此處示範驗證設定後中文與負號都正常。

In [ ]:
# 概念：畫一張含中文標題與負值座標的圖，確認字型設定有效
# 注意：字型族與 unicode_minus 已在本書第二個 cell 全域設定，這裡直接沿用
x = np.linspace(-5, 5, 100)         # 含負值的座標範圍
y = x ** 3                          # 立方，y 也會出現負值

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x, y)
ax.set_title('三次方曲線（含中文與負號）')   # 中文標題應正常顯示
ax.set_xlabel('輸入值')
ax.set_ylabel('輸出值')
ax.axhline(0, color='gray', linewidth=0.8)   # 畫 y=0 水平線，凸顯負區
plt.show()
plt.close(fig)

print('目前字型族設定：', plt.rcParams['font.family'])
print('unicode_minus：', plt.rcParams['axes.unicode_minus'])

## 練習

以下三題由淺到深，皆為複合型但簡短。資料一律自己用 numpy 或 list 造，不讀外部檔。

In [ ]:
# TODO 1 ·（簡單）建築樓高分布折線圖（整合：物件式介面 object-oriented interface + 折線圖 plot）
#   情境：用一串自造的建築樓高 height（公尺）資料，畫隨建築編號變化的樓高折線圖。
#   要求：
#     1. 用 numpy 或 list 自造 10 棟建築的樓高數值（如 12、24、36… 公尺）。
#     2. 以物件式介面 fig, ax = plt.subplots() 建立一張圖。
#     3. 在 ax 上用 ax.plot 畫樓高折線，並設中文標題與軸標籤（x 軸：建築編號，y 軸：樓高（公尺））。
#   驗收：應看到一條由左到右、隨建築編號起伏的樓高折線圖，標題與軸標籤皆為中文。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）街廓建築概況四格圖（整合：物件式介面 + 常用圖型 + 圖面標註 + 中文字型 font）
#   情境：把一個街廓 block 內多棟建築的指標整理成一張四格對照圖並存檔。
#   要求：
#     1. 用 numpy 自造同一批建築的樓高 height、樓層數 floors、占地面積 footprint area、容積率 FAR 等數列。
#     2. 以物件式介面建 2x2 子座標，四格分別畫折線圖 plot、長條圖 bar、散點圖 scatter、直方圖 histogram
#        （例如散點看樓高與樓層數關係、直方看 FAR 分布）。
#     3. 為每格加上中文標題與軸標籤（字型已於本書第二個 cell 設定）。
#     4. 用 tight_layout 排版後以指定 dpi savefig 存檔，最後 close 釋放資源。
#   驗收：應看到一張 2x2、四種圖型並列、中文標註正常的街廓概況圖檔，且程式結束未殘留未關閉的圖。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）政策前後容積比較與街廓示意圖
#   （整合：物件式介面 + 常用圖型 + 幾何多邊形繪製 + 座標控制 set_aspect equal + 圖面標註 + 中文字型 font）
#   情境：比較同一街廓在容積率政策調整（套用一個高度乘數）前後的樓地板面積 GFA，並畫街廓內建築平面示意。
#   要求：
#     1. 用 numpy 自造每棟建築的占地面積 footprint area 與容積率 FAR，算出現況 GFA；
#        再對 FAR 套一個高度乘數（如 1.2）得到政策後 GFA。
#     2. 左圖以長條圖 bar 並列比較各棟政策前後 GFA，並用 ax.text 標出總 GFA 增幅（聚合 aggregation 後相加比較）。
#     3. 右圖用 patches.Rectangle 依各棟占地面積在網格 cell 排平面方塊，
#        以 collections.LineCollection 畫街廓邊界，並用 set_aspect('equal') 維持比例、axis off 關閉座標軸。
#     4. 全圖中文字型與負號正常，加上總標題與圖例 legend。
#   驗收：左側為政策前後 GFA 並列長條與標出的總增幅文字；右側為等比例、不變形且關閉座標軸的街廓平面示意圖。
# TODO: 在這裡完成


## 小結

- matplotlib 有狀態式與物件式兩種介面；物件式以 `fig, ax = plt.subplots()` 取得畫布與座標系，可控可重用，是正式作圖首選。
- 輸出畫素由 figsize 與 dpi 共同決定；批次作圖要用 `close` 釋放記憶體、`tight_layout` 避免標籤被裁切。
- 依資料形態選圖：折線看趨勢、散點看分布、長條比類別、直方看數值分布。
- patches 與 collections 能畫幾何形狀，`set_aspect('equal')` 是維持形狀不變形的關鍵。
- 標題、軸標籤、legend、text、grid 讓圖能被獨立讀懂；legend 依賴 label，text 用資料座標定位。
- 用 font_manager 註冊字型並透過 rcParams 設為全域，再關閉 unicode_minus，中文與負號才能正常顯示。